In [0]:
#Silver Layer Transformation for both the trip data and zonelookup data 

df = spark.readStream.table("nycyellow.bronze.nyctripdata_raw")


In [0]:
#display(df.describe())


In [0]:
# from pyspark.sql.functions import  col, lit, unix_timestamp,to_timestamp,date_format


# #Checking if there is any relationship between the payment types, passenger count and trip distance
# display(df.select(["payment_type","passenger_count","trip_distance","tpep_pickup_datetime","tpep_dropoff_datetime","fare_amount","tip_amount","mta_tax","total_amount"])
#             .withColumn("duration",date_format(to_timestamp(unix_timestamp(col("tpep_dropoff_datetime"))-unix_timestamp(col("tpep_pickup_datetime"))),"HH:mm:ss"))
#             .filter((df.passenger_count == 0) & (df.total_amount < 0))
#             )

# # After analysis I have observed that if passenger count is 0 and total amount is negative,  could be a geniune trip cancellation and there are about 40 such records which we can drop           

In [0]:
# #Checking how many records are there where passenger count is greater than 0 and total amount is negative 
# display(df.select(["payment_type","passenger_count","trip_distance","tpep_pickup_datetime","tpep_dropoff_datetime","fare_amount","tip_amount","mta_tax","total_amount"])
#             .withColumn("duration",date_format(to_timestamp(unix_timestamp(col("tpep_dropoff_datetime"))-unix_timestamp(col("tpep_pickup_datetime"))),"HH:mm:ss"))
#             .filter((df.passenger_count > 0) & (df.total_amount < 0) & (df.trip_distance == 0) & (col("duration") < "00:10:00"))
#             )

In [0]:
# # In this step we are checking if there are any records with trip distance as 0 and total amount as negative
# # After analysing the data we have observed that there are about 5000+ such records which we can drop
# count = (
#     df.withColumn("duration",date_format(to_timestamp(unix_timestamp(col("tpep_dropoff_datetime"))-unix_timestamp(col("tpep_pickup_datetime"))),"HH:mm:ss"))
#     .filter(
#         (col("passenger_count") > 0) &
#         (col("total_amount") < 0) &
#         (col("trip_distance") == 0)
#     )
#     .count()
# )

# print(count)

In [0]:
# #With this logic we are dropping 43K records from the dataset
# count = (df.filter((df.passenger_count > 0) & (df.total_amount < 0) & (df.payment_type == 4))
#         .count())
# display(count)
            

In [0]:
# count = (
#     df.withColumn("duration",date_format(to_timestamp(unix_timestamp(col("tpep_dropoff_datetime"))-unix_timestamp(col("tpep_pickup_datetime"))),"HH:mm:ss"))
#     .filter(
#         (col("passenger_count") > 0) &
#         (col("total_amount")< 0) &
#        (col("payment_type").isin (3,4))  
#     )
#     .count()
# )

# print(count)

In [0]:
#Cleaning the data by dropping the passenger count is 0 and total amount is negative and payment_type is not 4
#After cleaning around 2 million records are dropped 
from pyspark.sql.functions import abs, when,date_format,to_timestamp,unix_timestamp,col

df_clean = (df.withColumn("total_amount", when((df.passenger_count > 0) & (df.total_amount < 0) & (df.trip_distance > 0), abs(df.total_amount)).otherwise(df.total_amount))
                .select("VendorID","passenger_count","trip_distance","PULocationID","DOLocationID","payment_type","total_amount","fare_amount","tip_amount","tpep_pickup_datetime","tpep_dropoff_datetime")
                .withColumn("trip_duration",date_format(to_timestamp(unix_timestamp(col("tpep_dropoff_datetime"))-unix_timestamp(col("tpep_pickup_datetime"))),"HH:mm:ss"))
                .filter((df.passenger_count > 0) | (df.trip_distance > 0) )
            )


In [0]:
# After cleaning I have observed there are still negative values in the total amount
#display(df_clean.filter(col("total_amount")<0))

In [0]:
df_final = (df_clean.filter((col("total_amount")>=0) & (col("fare_amount")>=0) & (col("tip_amount")>=0))
                    .withColumn("payment_status",when((col("payment_type") ==  0),"Flex Fair Trip")
                                                .when((col("payment_type") ==  1),"Credit Card")
                                                .when((col("payment_type") ==  2),"No charge")      
                                                .when((col("payment_type") ==  3),"Dispute")        
                                                .when((col("payment_type") ==  4),"Unknown")        
                                                .otherwise("Other"))
                    .withColumn("formated_pickup_date", date_format(col("tpep_pickup_datetime"),"yyyy-MM-dd").cast("date"))
                    .withColumn("formated_dropoff_date", date_format(col("tpep_dropoff_datetime"),"yyyy-MM-dd").cast("date"))
                    )


#display(df_final.describe())


In [0]:
#Getting the static table data to dataframe to join with the cleaned dataset 
from delta.tables import DeltaTable
df_zone_lookup_raw = spark.read.table("nycyellow.bronze.nyctripdata_zone_lookup")

if spark.catalog.tableExists("nycyellow.silver.zone_lookup"):
    zone_table = DeltaTable.forName(spark, "nycyellow.silver.zone_lookup")
    (zone_table.alias("target")
        .merge(df_zone_lookup_raw.alias("source"), "target.LocationID = source.LocationID")
        .whenMatchedUpdate(set={"Zone": "source.Zone", "Borough": "source.Borough", "service_zone": "source.service_zone"})
        .whenNotMatchedInsertAll()
        .execute())
else:
    df_zone_lookup_raw.write.format("delta").saveAsTable("nycyellow.silver.zone_lookup")

df_zone_lookup = spark.read.table("nycyellow.silver.zone_lookup")  

In [0]:
#Joining the final dataset with the zonelookup dataset 

from pyspark.sql.functions import col

df_combine = (df_final.join(df_zone_lookup,df_final.PULocationID == df_zone_lookup.LocationID,"left").select(df_final["*"],df_zone_lookup.Borough.alias("pickup_borough"),
        df_zone_lookup.Zone.alias("pickup_zone"),
        df_zone_lookup.service_zone.alias("pickup_service_zone")))


df_combine = df_combine.join(df_zone_lookup,df_combine.DOLocationID == df_zone_lookup.LocationID,"left").select(df_combine["*"],df_zone_lookup.Borough.alias("dropoff_borough"),
        df_zone_lookup.Zone.alias("dropoff_zone"),
        df_zone_lookup.service_zone.alias("dropoff_service_zone"))


#display(df_combine)


In [0]:
#Further refining as after adding the pick up zone and drop off zone I have observed that the location is same and the trip distance is less than 1 mile
df_combine_final = df_combine.filter(df_combine.trip_distance >= 1)
#display(df_combine_final.count())

In [0]:
#writing to the silver table
from delta.tables import DeltaTable

def upsert_to_silver(microBatchDF, batchId):
    if spark.catalog.tableExists("nycyellow.silver.cleaned_trip_data_with_zones"):
        silver_table = DeltaTable.forName(spark, "nycyellow.silver.cleaned_trip_data_with_zones")
        (silver_table.alias("target")
            .merge(
                microBatchDF.alias("source"),
                """target.tpep_pickup_datetime = source.tpep_pickup_datetime
                   AND target.tpep_dropoff_datetime = source.tpep_dropoff_datetime
                   AND target.PULocationID = source.PULocationID
                   AND target.DOLocationID = source.DOLocationID"""
            )
            .whenNotMatchedInsertAll()
            .execute())
    else:
        # first run — table doesn't exist yet, just create it
        microBatchDF.write.format("delta").saveAsTable("nycyellow.silver.cleaned_trip_data_with_zones")

(df_combine_final.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", "/Volumes/nycyellow/landing/raw/checkpoint/_checkpoint_silver")
    .trigger(availableNow=True)
    .start()
    .awaitTermination())